In [0]:
raw_path        = "abfss://raw@stockmarket.dfs.core.windows.net/stocks/"
bronze_path     = "abfss://bronze@stockmarket.dfs.core.windows.net/stocks/"
checkpoint_path = "abfss://bronze@stockmarket.dfs.core.windows.net/checkpoints/stocks/"
schema_path     = "abfss://bronze@stockmarket.dfs.core.windows.net/schema/stocks/"


df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .option("cloudFiles.schemaLocation", schema_path) \
    .option("cloudFiles.inferColumnTypes", "true") \
    .option("cloudFiles.useManagedFileEvents", "false") \
    .load(raw_path)

query = df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_path) \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .start(bronze_path)

query.awaitTermination()



